# 🩺 Semantic Segmentation with U-Net: Oxford-IIIT Pets

This notebook implements semantic segmentation using a compact **U-Net** in TensorFlow/Keras.

## Table of Contents
1. Semantic Segmentation Overview
2. U-Net Architecture
3. Dataset Loading
4. Mask Preprocessing
5. Model Training
6. Dice and IoU Metrics
7. Predicted Mask Visualization
8. Practical Applications
9. Key Learnings, Interview Questions, Common Mistakes, Further Reading


## 1. Semantic Segmentation Overview

Semantic segmentation assigns a class label to every pixel. Instead of asking “what is in the image?”, segmentation asks “which pixels belong to each object or region?”

```mermaid
flowchart LR
    A[Input image] --> B[Encoder: downsample]
    B --> C[Bottleneck]
    C --> D[Decoder: upsample]
    D --> E[Pixel-wise mask]
```


In [ ]:
# Reproducibility and common imports
import os
import random
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [ ]:
try:
    import tensorflow as tf
    import tensorflow_datasets as tfds
except ImportError:
    %pip install -q tensorflow tensorflow-datasets
    import tensorflow as tf
    import tensorflow_datasets as tfds

tf.random.set_seed(SEED)


## 2. Load Oxford-IIIT Pet Segmentation Dataset


In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 16

(ds_train, ds_test), info = tfds.load(
    "oxford_iiit_pet",
    split=["train", "test"],
    with_info=True,
)
print(info)


## 3. Preprocess Images and Masks

The trimap mask labels are converted to zero-based classes. Pixel labels represent pet, border, and background regions.


In [ ]:
def normalize(input_image, input_mask):
    input_image = tf.cast(input_image, tf.float32) / 255.0
    input_mask = tf.cast(input_mask, tf.int32) - 1
    return input_image, input_mask

def load_image(datapoint):
    input_image = tf.image.resize(datapoint["image"], IMG_SIZE)
    input_mask = tf.image.resize(
        datapoint["segmentation_mask"],
        IMG_SIZE,
        method=tf.image.ResizeMethod.NEAREST_NEIGHBOR,
    )
    return normalize(input_image, input_mask)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = ds_train.map(load_image, num_parallel_calls=AUTOTUNE).cache().shuffle(1000, seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = ds_test.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)


In [ ]:
def display_sample(display_list, title_list=None):
    plt.figure(figsize=(12, 4))
    for i, item in enumerate(display_list):
        ax = plt.subplot(1, len(display_list), i + 1)
        if title_list:
            ax.set_title(title_list[i])
        plt.imshow(tf.keras.utils.array_to_img(item) if item.shape[-1] == 3 else tf.squeeze(item), cmap="viridis")
        plt.axis("off")
    plt.tight_layout()

for images, masks in train_ds.take(1):
    display_sample([images[0], masks[0]], ["Image", "Ground Truth Mask"])


## 4. Build U-Net

U-Net combines an encoder path for context with a decoder path for precise localization. Skip connections recover spatial detail lost during downsampling.


In [ ]:
def conv_block(x, filters):
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    return x

def encoder_block(x, filters):
    c = conv_block(x, filters)
    p = tf.keras.layers.MaxPooling2D()(c)
    return c, p

def decoder_block(x, skip, filters):
    x = tf.keras.layers.Conv2DTranspose(filters, 2, strides=2, padding="same")(x)
    x = tf.keras.layers.Concatenate()([x, skip])
    return conv_block(x, filters)

def build_unet(input_shape=(128, 128, 3), num_classes=3):
    inputs = tf.keras.Input(shape=input_shape)
    s1, p1 = encoder_block(inputs, 32)
    s2, p2 = encoder_block(p1, 64)
    s3, p3 = encoder_block(p2, 128)
    b = conv_block(p3, 256)
    d1 = decoder_block(b, s3, 128)
    d2 = decoder_block(d1, s2, 64)
    d3 = decoder_block(d2, s1, 32)
    outputs = tf.keras.layers.Conv2D(num_classes, 1, activation="softmax")(d3)
    return tf.keras.Model(inputs, outputs, name="pet_unet")

model = build_unet(num_classes=3)
model.summary()


## 5. Loss Functions and Metrics

Sparse categorical cross-entropy trains pixel-wise multi-class predictions. Dice and IoU are overlap metrics commonly used for segmentation quality.


In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(tf.squeeze(y_true, axis=-1), tf.int32)
    y_true = tf.one_hot(y_true, depth=3)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2])
    denominator = tf.reduce_sum(y_true + y_pred, axis=[1, 2])
    dice = (2.0 * intersection + smooth) / (denominator + smooth)
    return tf.reduce_mean(dice)

def mean_iou_metric(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(tf.squeeze(y_true, axis=-1), tf.int32)
    y_true = tf.one_hot(y_true, depth=3)
    y_pred = tf.one_hot(tf.argmax(y_pred, axis=-1), depth=3)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2])
    union = tf.reduce_sum(y_true + y_pred, axis=[1, 2]) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return tf.reduce_mean(iou)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[dice_coefficient, mean_iou_metric],
)


In [ ]:
history = model.fit(train_ds, validation_data=test_ds, epochs=10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ["dice_coefficient", "mean_iou_metric"]):
    ax.plot(history.history[metric], label=f"train_{metric}", linewidth=2)
    ax.plot(history.history[f"val_{metric}"], label=f"val_{metric}", linewidth=2)
    ax.set_title(metric.replace("_", " ").title())
    ax.set_xlabel("Epoch")
    ax.legend()
plt.tight_layout()


## 6. Visualize Predicted Masks


In [ ]:
def create_mask(pred_mask):
    pred_mask = tf.argmax(pred_mask, axis=-1)
    pred_mask = pred_mask[..., tf.newaxis]
    return pred_mask[0]

for images, masks in test_ds.take(3):
    pred_mask = model.predict(images[:1])
    display_sample([images[0], masks[0], create_mask(pred_mask)], ["Image", "Ground Truth", "Prediction"])


## Practical Applications

- Medical image analysis
- Autonomous driving scene understanding
- Background replacement
- Satellite image mapping
- Industrial defect localization

## Key Learnings

- Segmentation is pixel-level classification.
- U-Net skip connections preserve detail.
- Mask preprocessing must keep integer labels intact.
- Dice and IoU are more informative than raw pixel accuracy.

## Interview Questions

1. Why does U-Net use skip connections?
2. What is the difference between semantic and instance segmentation?
3. Why should masks be resized with nearest-neighbor interpolation?
4. How do Dice and IoU differ?

## Common Mistakes

- Resizing masks with bilinear interpolation.
- Treating segmentation masks as RGB photos.
- Optimizing pixel accuracy on highly imbalanced masks.
- Forgetting to inspect predicted masks visually.

## Further Reading

- U-Net paper: https://arxiv.org/abs/1505.04597
- TensorFlow segmentation tutorial: https://www.tensorflow.org/tutorials/images/segmentation
- Oxford-IIIT Pet dataset: https://www.robots.ox.ac.uk/~vgg/data/pets/
